In [94]:
import json
import pandas as pd
import requests
import time
import re
import multiprocessing
import math
from transformers import BertModel, BertConfig, BertTokenizer
import torch

In [76]:
API_Key = '3wTCnCmoynBrzneDZwMROjO5'
Secret_Key = 'WpQ31wCwerhD5F3ESmUx8DuRzNMlRS29'

## 处理 course.json

In [78]:

def get_access_token():
    """
    使用 API Key，Secret Key 获取access_token，替换下列示例中的应用API Key、应用Secret Key
    """
    url = "https://aip.baidubce.com/oauth/2.0/token?grant_type=client_credentials&client_id=" + API_Key + "&client_secret=" + Secret_Key + ""
    payload = json.dumps("")
    headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data=payload)
    return response.json().get("access_token")


def text_to_vector(text_list):
    url = "https://aip.baidubce.com/rpc/2.0/ai_custom/v1/wenxinworkshop/embeddings/embedding-v1?access_token=" + get_access_token()
    payload = json.dumps({
        "input": text_list
    })
    headers = {
        'Content-Type': 'application/json'
    }

    response = requests.request("POST", url, headers=headers, data=payload)
    response_data = json.loads(response.text)
    em = response_data
    return em

In [151]:
course_data = []

#  ["id","name","field","prerequisites","about"]


# 打开JSON文件
with open('F:\course.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        course = json.loads(line)
        Id = course["id"]
        name = course["name"]
        field = ','.join(list(course["field"]))
        prerequisites = course["prerequisites"]
        about = str(course["about"]).replace(' ', '').replace('\t', '').replace('\n', '')
        course_data.append([Id,name,field,prerequisites,about])

In [152]:
len(course_data)

3781

In [153]:
df = pd.DataFrame(course_data,columns=["id","name","field","prerequisites","about"])

In [154]:
df = df[df['about'].notna() & (df['about'] != '')].reset_index(drop=True)

In [155]:
df = df[df['id'].notna() & (df['id'] != '')].reset_index(drop=True)

In [156]:
about = df["about"].tolist()

### 文本转向量

In [162]:
### 百度的模型最大只能调用16个  按照16个来切分
chunked_list = [about[i:i+16] for i in range(0, len(about), 16)]

In [163]:
embedding = []

In [164]:
for chunk in chunked_list:
    # chunk = [s for s in  chunk if s and len(s.strip()) > 0]
    # chunk = [s.strip() for s in chunk]    
    embedding_t = text_to_vector(chunk)["data"]
    for e in embedding_t:
        embedding.append(e["embedding"])
    # time.sleep(5)

In [167]:
df["embedding"]= embedding

In [177]:
df.to_csv('F:\course.csv', index=False)

## 处理 school.json

In [170]:
school_data = []
# 打开JSON文件
with open('F:\mooccubex\json\school.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        school = json.loads(line)
        Id = school["id"]
        name = school["name"]
        name_en = school["name_en"]
        sign = school["sign"]    
        about = str(school["about"]).replace(' ', '').replace('\t', '').replace('\n', '')
        motto = str(school["motto"]).replace(' ', '').replace('\t', '').replace('\n', '')
        school_data.append([Id,name,name_en,sign,about,motto])

In [173]:
df_school = pd.DataFrame(school_data,columns=["id","name","name_en","sign","about","motto"])

In [178]:
df_school.to_csv('F:\mooccubex\csv\school.csv', index=False)

## 处理teacher.json

In [193]:
teacher_data = []
# 打开JSON文件
with open('F:\\mooccubex\\json\\teacher.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        teacher = json.loads(line)
        Id = teacher["id"]
        name = teacher["name"]
        name_en = teacher["name_en"]  
        about = str(teacher["about"]).replace(' ', '').replace('\t', '').replace('\n', '')
        job_title = str(teacher["job_title"]).replace(' ', '').replace('\t', '').replace('\n', '')
        org_name = str(teacher["org_name"]).replace(' ', '').replace('\t', '').replace('\n', '')
        teacher_data.append([Id,name,name_en,about,job_title,org_name])

In [194]:
len(teacher_data)

17018

In [195]:
df_teacher = pd.DataFrame(teacher_data,columns=["id","name","name_en","about","job_title","org_name"])

In [196]:
df_teacher.to_csv('F:\\mooccubex\\csv\\teacher.csv', index=False)

## 处理course-field.json

In [192]:
course_field_data = []
# 打开JSON文件
with open('F:\\mooccubex\\json\\course-field.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        course_field = json.loads(line)
        course_id = course_field["course_id"]
        course_name = course_field["course_name"]
        field = str(course_field["field"]).replace(' ', '').replace('\t', '').replace('\n', '')
        course_field_data.append([course_id,course_name,field])

In [197]:
df_cf = pd.DataFrame(course_field_data,columns=["course_id","course_name","field"])

In [198]:
df_cf.to_csv('F:\\mooccubex\\csv\\course-field.csv', index=False)

## 处理user.json

In [200]:
user_data = []
# 打开JSON文件
with open('F:\\mooccubex\\json\\user.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        user = json.loads(line)
        Id = user["id"]
        name = user["name"]
        gender = user["gender"]
        school = user["school"]
        year_of_birth = user["year_of_birth"]
        course_order = str(user["course_order"]).replace(' ', '').replace('\t', '').replace('\n', '')
        enroll_time = str(user["enroll_time"]).replace(' ', '').replace('\t', '').replace('\n', '')
        user_data.append([Id,name,gender,school,year_of_birth,course_order,enroll_time])

In [201]:
df_user = pd.DataFrame(user_data,columns=["id","name","gender","school","year_of_birth","course_order","enroll_time"])

In [203]:
df_user.to_csv('F:\\mooccubex\\csv\\user.csv', index=False)

## 处理problem

In [6]:
problem_data = []

# 打开JSON文件
with open('/home/wsf/Desktop/mooccubex/json/problem.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        problem = json.loads(line)
        Id = problem["problem_id"]
        exercise_id = problem["exercise_id"]
        language = problem["language"]
        title = problem["title"]
        content = problem["content"]
        option = problem["option"]
        answer = problem["answer"]
        score = problem["score"]
        Type = problem["type"]
        typetext = problem["typetext"]
        problem_data.append([Id,exercise_id,language,title,content,option,answer,score,Type,typetext])

In [19]:
df_problem = pd.DataFrame(problem_data,columns=["id","exercise_id","language","title","content","option","answer","score","type","typetext"])

In [20]:
df_problem.head(5)

,id,exercise_id,language,title,content,option,answer,score,type,typetext
0,1730,Ex_856,Chinese,第一课 导论与三家分晋--习题,1、《资治通鉴》卷1记载：智宣子将以瑶为后，智果曰：“……瑶之贤于人者五，其不逮者一也。美鬓...,"{'A': '武艺超群，精通射御之术', 'B': '礼贤下士，虚怀若谷', 'C': '反...","[""B""]",1.0,1,单选题
1,1731,Ex_856,Chinese,第一课 导论与三家分晋--习题,2、《资治通鉴》是一部____史书。,"{'A': '纪传体', 'B': '编年体', 'C': '纪事本末体', 'D': '国...","[""B""]",1.0,1,单选题
2,1732,Ex_856,Chinese,第一课 导论与三家分晋--习题,3、《资治通鉴》原名____，后由____赐名“资治通鉴”。,"{'A': '《通鉴》；宋神宗', 'B': '《通志》；宋徽宗', 'C': '《通鉴》；...","[""D""]",1.0,1,单选题
3,1733,Ex_856,Chinese,第一课 导论与三家分晋--习题,4、“三家分晋”中“三家”具体指：,"{'A': '魏赵韩', 'B': '魏韩智', 'C': '赵韩智', 'D': '魏赵智'}","[""A""]",1.0,1,单选题
4,1734,Ex_856,Chinese,第一课 导论与三家分晋--习题,5、智伯联合韩、魏的军队攻打赵氏时，赵襄子选择退守的阵地是：,"{'A': '邯郸', 'B': '长子', 'C': '晋阳', 'D': '皋狼'}","[""C""]",1.0,1,单选题


### 清洗数据

In [10]:
unique_typetext = df_problem['typetext'].unique()

In [11]:
unique_typetext

array(['单选题', '判断题', '多选题', '填空题', '主观题', '投票题', '编程题'], dtype=object)

In [21]:
#删除掉 “title  option answer ”列
df_problem = df_problem.drop('title', axis=1)
df_problem = df_problem.drop('option', axis=1)
df_problem = df_problem.drop('answer', axis=1)


In [34]:
### 清洗掉 content 中的序号
content = df_problem['content']
pattern = r'^\s*\d+[\u3001、]'
# 使用str.replace方法替换匹配到的部分为空字符串
content_cleaned = content.str.replace(pattern, '', regex=True)
df_problem['content'] = content_cleaned

In [36]:
### 清洗掉 content 中是文件的数据
file_extensions = ['.txt', '.jpg', '.mp4', '.wav', '.png', '.pdf', '.docx', '.xls','.doc','.pptx','.ppt']
pattern1 = '|'.join(file_extensions).replace('.', '\\.')
df_problem = df_problem[~df_problem['content'].str.contains(pattern1, regex=True)]


In [40]:
### 清洗掉 socre 中值为NaN的行
df_problem.dropna(subset=['score'], inplace=True)

/opt/miniconda3/lib/python3.7/site-packages/pandas/util/_decorators.py:311: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return func(*args, **kwargs)


In [56]:
### 在清洗掉content 部分数据
df_problem = df_problem.loc[~((df_problem['language'] == 'Chinese') & (df_problem['content'].str.contains(r'\[填空1\]')))]

In [60]:
### 在清洗掉content 部分数据
df_problem = df_problem.loc[~((df_problem['language'] == 'English') & (df_problem['content'].str.contains(r'\$underset')))]

In [68]:
### 在清洗掉content 部分数据
df_problem = df_problem.loc[~((df_problem['language'] == 'English') & (df_problem['content'].str.contains(r'\$')))]

In [69]:
df_problem

,id,exercise_id,language,content,score,type,typetext
0,1730,Ex_856,Chinese,《资治通鉴》卷1记载：智宣子将以瑶为后，智果曰：“……瑶之贤于人者五，其不逮者一也。美鬓长大...,1.0,1,单选题
1,1731,Ex_856,Chinese,《资治通鉴》是一部____史书。,1.0,1,单选题
2,1732,Ex_856,Chinese,《资治通鉴》原名____，后由____赐名“资治通鉴”。,1.0,1,单选题
3,1733,Ex_856,Chinese,“三家分晋”中“三家”具体指：,1.0,1,单选题
4,1734,Ex_856,Chinese,智伯联合韩、魏的军队攻打赵氏时，赵襄子选择退守的阵地是：,1.0,1,单选题
...,...,...,...,...,...,...,...
2451996,8420963,Ex_8621996,English,The endorsor of of Insurance Policy is Shangha...,2.0,6,判断题
2451997,8420964,Ex_8621996,English,CREDIT NUMBER AND NAME OF ISSUING BANK are no ...,2.0,6,判断题
2451998,8420965,Ex_8621996,English,The Bill of Lading must be endorsed in blank b...,2.0,6,判断题
2451999,8420966,Ex_8621996,English,The goods description of goods in the Bill of ...,2.0,6,判断题


### content 转向量

In [70]:
### 先把中文和英分割出来，分别调用不同的模型来转成向量
df_problem_zh = df_problem[df_problem['language'] == 'Chinese']
df_problem_en = df_problem[df_problem['language'] == 'English']

In [71]:
df_problem_zh_content = df_problem_zh["content"]
df_problem_en_content = df_problem_en["content"]

In [79]:
def text_to_vector_zh_1024_dim(text_list):
    url = "https://aip.baidubce.com/rpc/2.0/ai_custom/v1/wenxinworkshop/embeddings/bge_large_zh?access_token=" + get_access_token()
    payload = json.dumps({
        "input": text_list
    })
    headers = {
        'Content-Type': 'application/json'
    }

    response = requests.request("POST", url, headers=headers, data=payload)
    response_data = json.loads(response.text)
    em = response_data
    return em

In [86]:
df_problem_zh_content_list = df_problem_zh_content.tolist()

#### 多进程 文本转向量

In [96]:
def split_list_into_parts(lst, num_parts):
    avg = len(lst) // num_parts
    remainder = len(lst) % num_parts
    parts = [lst[i * avg + min(i, remainder):(i + 1) * avg + min(i + 1, remainder)] for i in range(num_parts)]
    return parts

In [99]:
### 12 core cpu, data split to 12
df_problem_zh_content_list_splited = split_list_into_parts(df_problem_zh_content_list, 12)

In [136]:
# 创建一个共享的列表
manager = multiprocessing.Manager()
content_vector = manager.list([None]*12)

In [157]:
def process_element(args):
    # 解包元组，获取索引和元素
    index, element = args
    print(f'启动进程 {index}')
    element_chunked = [element[i:i+16] for i in range(0, len(element), 16)]
    embedding = []
    for chunk in element_chunked:
#         print(text_to_vector(chunk))
        embedding_t = text_to_vector(chunk)["data"]
        for e in embedding_t:
                embedding.append(e["embedding"])
    content_vector[index] = embedding


In [158]:
with multiprocessing.Pool(processes=12) as pool:
        # 使用map方法将process_element函数应用到列表中的每个元素上
        pool.map(process_element, [(i, element) for i, element in enumerate(df_problem_zh_content_list_splited)])

启动进程 0
启动进程 1
启动进程 2
启动进程 3
启动进程 4
启动进程 5
启动进程 6
启动进程 7
启动进程 8
启动进程 9
启动进程 10
启动进程 11


KeyError: 'data'

#### 单进程

In [ ]:
df_problem_zh_content_list_chunked = [df_problem_zh_content_list[i:i+16] for i in range(0, len(df_problem_zh_content_list), 16)]


content_vector = []
index = 1 

for chunk in df_problem_zh_content_list_chunked:
    response_data = text_to_vector_zh_1024_dim(chunk)
    if 'error_code' in response_data:
        for i in range(16):
            content_vector.append([])
    else:
        embedding = response_data["data"]
        for e in embedding:
            content_vector.append(e["embedding"])
    
    print(f'正在处理：{index}')
    index = index + 1
    time.sleep(random.uniform(0, 1.5))

df_problem_zh["content_vector"] = content_vector

df_problem_zh.to_csv('/home/wsf/python/problem_zh.csv', index=False)

## 处理comment

In [177]:
comment_data = []

# 打开JSON文件
with open('/home/wsf/Desktop/mooccubex/json/comment.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        comment = json.loads(line)
        Id = comment["id"]
        user_id = comment["user_id"]
        text = comment["text"].replace(' ', '').replace('\t', '').replace('\n', '')
        create_time = comment["create_time"]
        comment_data.append([Id,user_id,text,create_time])

In [178]:
df_comment = pd.DataFrame(comment_data,columns=["id","user_id","text","create_time"])

In [179]:
### 删除 text 列中字符串数少于10个字的行
df_comment['text_length'] = df_comment['text'].str.len()

df_comment = df_comment[df_comment['text_length'] >= 10]
df_comment = df_comment[df_comment['text_length'] <= 128]

In [180]:
df_comment = df_comment.drop('text_length', axis=1)

In [181]:
### 删除text列数据重复频率大于1的行

df_comment = df_comment.drop_duplicates(subset='text', keep=False)


In [182]:
### 清洗掉 text 中的序号
text = df_comment['text']
pattern2 = r'^\s*\d+[\u3001、]'
# 使用str.replace方法替换匹配到的部分为空字符串
text_cleaned = text.str.replace(pattern2, '', regex=True)
df_comment['text'] = text_cleaned

In [206]:
t = [len(t) for t in text]

In [207]:
count_less_than_4000 = sum(1 for item in t if item <= 32)
count_less_than_4000

2

In [187]:
df_comment

,id,user_id,text,create_time
10,Cm_23,10031509,讨论区无直接粘贴功能，无@老师或学生提醒指定人员回答功能,2019-08-10 15:22:06
12,Cm_29,10031666,用手机4G热点看视频，速度相对还可以,2019-08-12 12:20:04
13,Cm_31,10031666,图片显示全幅速度比较慢,2019-08-12 12:35:17
16,Cm_36,10031572,讨论区自己发布的话题无法进行重新编辑8、作答习题时，每个题目需要分别提交答案，不能一次性提交...,2019-08-12 14:08:32
22,Cm_47,10031594,补充一点：页面的“返回”逻辑太复杂,2019-08-12 16:18:56
...,...,...,...,...
8395064,Cm_17043931,32344499,人工智能，大数据智能化,2020-11-19 03:42:02
8395068,Cm_17043936,5125147,字节跳动。抓住了短视频红利的机会，善于发现创业机遇。独立研发的“今日头条”客户端，通过海量信...,2020-11-19 03:42:49
8395071,Cm_17043945,4606463,合理安排时间，做到不浪费，注重效率,2020-11-19 03:44:04
8395074,Cm_17043948,36776316,企业领导者个人特质对企业初期发展的影响绝不仅仅存在于团队的选择和建设上,2020-11-19 03:44:19


In [188]:
df_comment.to_csv('/home/wsf/Desktop/mooccubex/csv/comment_clear.csv', index=False)

In [197]:
rows_per_chunk = len(df_comment) // 10

In [201]:
for i in range(10):
    # 计算每部分的起始和结束行索引
    start = i * rows_per_chunk
    # 确保最后一个部分包含剩余的所有行
    end = (i + 1) * rows_per_chunk if i < 9 else len(df_comment)
    
    # 选择DataFrame的一部分
    chunk = df_comment.iloc[start:end]
    
    # 存储为CSV文件
    chunk.to_csv(f'/home/wsf/Desktop/mooccubex/csv/comment/chunk_{i+1}.csv', index=False)

In [ ]:
def bert_define(path="/remote-home/cs_acmis_wsf/ai4dingo/bert_chinese_pretrained"):
    # 加载bert的tokenizer分词
    tokenizer = BertTokenizer.from_pretrained(path)
    # 加载预训练模型
    model_config = BertConfig.from_pretrained(path)
    model = BertModel.from_pretrained(path, config=model_config)
    return tokenizer, model

In [ ]:
for i in range(len(text)):
    t = text[i]
    if i % 100000 == 0 and i != 0:  # 检查是否是10000的倍数且不是第0次迭代
        print(f"处理到第 {(i // 100000)+1} 个10W")
        
    # text 作 tokenizer 分词
    token = tokenizer.tokenize(t)
    token = ['[CLS]'] + token + ['[SEP]']
    token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

    # 加padding补齐及segment、mask
    padding = [0] * (max_len - len(token_id))
    mask = [1] * len(token_id) + padding
    segment = [0] * len(token_id) + padding
    token_id = token_id + padding

    batch_token.append(token_id)
    batch_segment.append(segment)
    batch_mask.append(mask)

print("======文本处理完毕======")  

In [ ]:
batch_tensor_token = torch.tensor(batch_token)
batch_tensor_segment = torch.tensor(batch_segment)
batch_tensor_mask = torch.tensor(batch_mask)

text_vector = None

In [ ]:
print("======开始转换======") 
# text encode
with torch.no_grad():
    outputs = model(batch_tensor_token, token_type_ids=batch_tensor_segment, attention_mask=batch_tensor_mask)
    outputs = outputs[0][:, 0, :]  # 取cls向量
    print(outputs.shape)  #
    text_vector = outputs
